## ⚠️ Stop Loss Logic Reminder

**Stop Loss Priority:** When both a sell signal and stop loss are triggered on the same bar:
- **Stop loss takes priority** because it represents a forced exit due to loss limit
- The sell signal becomes irrelevant if we're already stopped out
- This reflects real trading where protective stops execute automatically

**Order of Operations per Bar:**
1. Check if stop loss triggered (intrabar price movement)
2. If stopped out → exit at stop loss price, position = 0
3. If not stopped → then evaluate buy/sell signals
4. Apply trading costs to all transactions

## 📊 1H Strategy - Ensemble Prediction with Stop Loss

This notebook implements a 1-hour trading strategy using an ensemble of the top 3 models by forward accuracy.

**Components:**
1. Load pre-trained models from `results/ETH_USD_1h_20260115_134213/`
2. Fetch live 1H data from yfinance
3. Calculate technical indicators (20 selected + 31 full features)
4. Generate ensemble predictions
5. Backtest with stop loss optimization

In [1]:
# Cell 3: Import Libraries
import pandas as pd
import numpy as np
import yfinance as yf
import joblib
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import TensorFlow/Keras for LSTM models
try:
    import tensorflow as tf
    from tensorflow import keras
    TF_AVAILABLE = True
    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not available - LSTM models will be skipped")

print("Libraries imported successfully")

TensorFlow version: 2.20.0
Libraries imported successfully


In [12]:
# Cell 4: Load Models from Results Folder
RESULTS_FOLDER = r"python_classes\results\ETH_USD_1h_20260221_051420"

# Load models registry
with open(os.path.join(RESULTS_FOLDER, "models_registry.json"), "r") as f:
    models_registry = json.load(f)

# Load features configuration
with open(os.path.join(RESULTS_FOLDER, "features.json"), "r") as f:
    features_config = json.load(f)

X_SELECTED_FEATURES = features_config["X_features"]
X_FULL_FEATURES = features_config["X_full_features"]

print(f"X Selected Features ({len(X_SELECTED_FEATURES)}): {X_SELECTED_FEATURES}")
print(f"\nX Full Features ({len(X_FULL_FEATURES)}): {X_FULL_FEATURES}")

# Get top 3 models by forward accuracy
leaderboard = models_registry["leaderboard_forward"]
top_3_models = leaderboard[:3]

print("\n=== Top 3 Models by Forward Accuracy ===")
for m in top_3_models:
    print(f"  Rank {m['rank']}: {m['model']} ({m['type']}) - Forward Acc: {m['forward_acc']:.4f}")

# Load the actual model files
MODELS_1h = {}
for model_info in top_3_models:
    model_name = model_info["model"]
    model_type = model_info["type"]
    
    if model_type == "sklearn":
        # Try .pkl first, then .joblib
        model_path = os.path.join(RESULTS_FOLDER, "models", f"{model_name}.pkl")
        if not os.path.exists(model_path):
            model_path = os.path.join(RESULTS_FOLDER, "models", f"{model_name}.joblib")
            
        if os.path.exists(model_path):
            MODELS_1h[model_name] = {
                "model": joblib.load(model_path),
                "type": model_type,
                "feature_set": model_info["feature_set"],
                "forward_acc": model_info["forward_acc"]
            }
            print(f"✓ Loaded {model_name}")
    elif model_type == "keras" and TF_AVAILABLE:
        model_path = os.path.join(RESULTS_FOLDER, "models", f"{model_name}.keras")
        if os.path.exists(model_path):
            loaded_model = keras.models.load_model(model_path)
            MODELS_1h[model_name] = {
                "model": loaded_model,
                "type": model_type,
                "feature_set": model_info["feature_set"],
                "forward_acc": model_info["forward_acc"]
            }
            print(f"✓ Loaded {model_name} (Keras)")

print(f"\nTotal models loaded: {len(MODELS_1h)}")

X Selected Features (20): ['Open', 'Volume', 'HTF_1d_Return_prev', 'RSI_20', 'ROC_10', 'CCI_20', 'MACD', 'MACD_hist', 'ATR_14', 'BW_20', 'STD_10', 'CMF_20', 'Log_Return', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_5', 'Volume_lag1', 'Volume_lag2']

X Full Features (31): ['Open', 'Close', 'Volume', 'HTF_1d_Close_prev', 'HTF_1d_Return_prev', 'HTF_1d_LogReturn_prev', 'EMA_20', 'SAR', 'RSI_20', 'ROC_10', 'CCI_20', 'MACD', 'MACD_signal', 'MACD_hist', 'ATR_14', 'BB_lower', 'BW_20', 'SMA_10', 'SMA_20', 'SMA_50', 'STD_10', 'STD_20', 'CMF_20', 'Log_Return', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_5', 'Volume_lag1', 'Volume_lag2']

=== Top 3 Models by Forward Accuracy ===
  Rank 1: LightGBM_Full (sklearn) - Forward Acc: 0.5301
  Rank 2: CNN_Full (keras) - Forward Acc: 0.5268
  Rank 3: RandomForest_Full (sklearn) - Forward Acc: 0.5242
✓ Loaded CNN_Full (Keras)

Total models loaded: 1


In [13]:
# Cell 5: Load 1H Data from yfinance
SYMBOL = "ETH-USD"
INTERVAL = "1h"

# Fetch 1h data - max 730 days for 1h interval
print(f"Fetching {SYMBOL} data with {INTERVAL} interval...")
DF_1H_RAW = yf.download(SYMBOL, period="60d", interval=INTERVAL, progress=False)

# Flatten multi-level columns if present
if isinstance(DF_1H_RAW.columns, pd.MultiIndex):
    DF_1H_RAW.columns = DF_1H_RAW.columns.get_level_values(0)

print(f"Data shape: {DF_1H_RAW.shape}")
print(f"Date range: {DF_1H_RAW.index[0]} to {DF_1H_RAW.index[-1]}")
DF_1H_RAW.tail()

Fetching ETH-USD data with 1h interval...
Data shape: (1385, 5)
Date range: 2025-11-29 19:00:00+00:00 to 2026-01-27 22:00:00+00:00


Price,Close,High,Low,Open,Volume
Datetime,,,,,
2026-01-27 18:00:00+00:00,2977.871094,2983.322021,2923.629150,2926.207520,0
2026-01-27 19:00:00+00:00,2975.716309,2986.180908,2962.686035,2974.156494,661317632
2026-01-27 20:00:00+00:00,3017.770508,3021.516357,2975.874023,2976.074219,2760249344
2026-01-27 21:00:00+00:00,3011.414062,3025.143555,3004.316162,3018.701172,1412038656
2026-01-27 22:00:00+00:00,3015.099121,3017.468506,3001.580078,3011.310791,1279655936


In [14]:
# Cell 6: Calculate Technical Indicators (31 features for 1H)
df = DF_1H_RAW.copy()

# Fetch Higher Timeframe Data for 1H (Daily)
print("Fetching Higher Timeframe Data (1d)...")
htf_1d = yf.download(SYMBOL, period="120d", interval="1d", progress=False)
if isinstance(htf_1d.columns, pd.MultiIndex):
    htf_1d.columns = htf_1d.columns.get_level_values(0)

# Calculate HTF 1d features
htf_1d['HTF_1d_Close'] = htf_1d['Close']
htf_1d['HTF_1d_Return'] = htf_1d['Close'].pct_change()
htf_1d['HTF_1d_LogReturn'] = np.log(htf_1d['Close'] / htf_1d['Close'].shift(1))

# Shift HTF data to avoid lookahead bias
htf_1d['HTF_1d_Close_prev'] = htf_1d['HTF_1d_Close'].shift(1)
htf_1d['HTF_1d_Return_prev'] = htf_1d['HTF_1d_Return'].shift(1)
htf_1d['HTF_1d_LogReturn_prev'] = htf_1d['HTF_1d_LogReturn'].shift(1)

# Merge HTF features into main dataframe
df = df.copy()
df['date_only'] = df.index.date
htf_1d_subset = htf_1d[['HTF_1d_Close_prev', 'HTF_1d_Return_prev', 'HTF_1d_LogReturn_prev']].copy()
htf_1d_subset.index = htf_1d_subset.index.date
df = df.merge(htf_1d_subset, left_on='date_only', right_index=True, how='left')
df = df.drop(columns=['date_only'])
df.index = DF_1H_RAW.index[:len(df)]

# Basic indicators
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['STD_10'] = df['Close'].rolling(window=10).std()
df['STD_20'] = df['Close'].rolling(window=20).std()

# Parabolic SAR (simplified)
df['SAR'] = df['Close'].ewm(span=2, adjust=False).mean()

# RSI
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(window=20).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=20).mean()
rs = gain / loss
df['RSI_20'] = 100 - (100 / (1 + rs))

# ROC
df['ROC_10'] = ((df['Close'] - df['Close'].shift(10)) / df['Close'].shift(10)) * 100

# CCI
tp = (df['High'] + df['Low'] + df['Close']) / 3
df['CCI_20'] = (tp - tp.rolling(window=20).mean()) / (0.015 * tp.rolling(window=20).std())

# MACD
ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema12 - ema26
df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_hist'] = df['MACD'] - df['MACD_signal']

# ATR
high_low = df['High'] - df['Low']
high_close = abs(df['High'] - df['Close'].shift())
low_close = abs(df['Low'] - df['Close'].shift())
tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df['ATR_14'] = tr.rolling(window=14).mean()

# Bollinger Bands
df['BB_middle'] = df['Close'].rolling(window=20).mean()
df['BB_std'] = df['Close'].rolling(window=20).std()
df['BB_lower'] = df['BB_middle'] - 2 * df['BB_std']
df['BW_20'] = (4 * df['BB_std']) / df['BB_middle']  # Bandwidth

# CMF (Chaikin Money Flow)
mfm = ((df['Close'] - df['Low']) - (df['High'] - df['Close'])) / (df['High'] - df['Low'])
mfv = mfm * df['Volume']
df['CMF_20'] = mfv.rolling(window=20).sum() / df['Volume'].rolling(window=20).sum()

# Log Return
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Lag features
for i in range(1, 6):
    df[f'Lag_{i}'] = df['Log_Return'].shift(i)

# Volume lags
df['Volume_lag1'] = df['Volume'].shift(1)
df['Volume_lag2'] = df['Volume'].shift(2)

# Drop NaN and keep copy
df = df.dropna()
DF_1H_MAX = df.copy()

print(f"\nProcessed data shape: {DF_1H_MAX.shape}")
print(f"Date range: {DF_1H_MAX.index[0]} to {DF_1H_MAX.index[-1]}")
print(f"\nAvailable columns: {list(DF_1H_MAX.columns)}")

Fetching Higher Timeframe Data (1d)...

Processed data shape: (1310, 35)
Date range: 2025-12-03 07:00:00+00:00 to 2026-01-27 22:00:00+00:00

Available columns: ['Close', 'High', 'Low', 'Open', 'Volume', 'HTF_1d_Close_prev', 'HTF_1d_Return_prev', 'HTF_1d_LogReturn_prev', 'EMA_20', 'SMA_10', 'SMA_20', 'SMA_50', 'STD_10', 'STD_20', 'SAR', 'RSI_20', 'ROC_10', 'CCI_20', 'MACD', 'MACD_signal', 'MACD_hist', 'ATR_14', 'BB_middle', 'BB_std', 'BB_lower', 'BW_20', 'CMF_20', 'Log_Return', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_5', 'Volume_lag1', 'Volume_lag2']


### load mean and std for normalization

In [ ]:
with open(os.path.join(RESULTS_FOLDER, "config.json"), "r") as f:
    config = json.load(f)

scaler = config["scaler"]
print(f"\nConfig loaded. Scaler keys: {list(scaler.keys())}")

# Dicts from config
x_mean_dict = scaler["X_mean"]
x_std_dict = scaler["X_std"]
x_full_mean_dict = scaler["X_full_mean"]
x_full_std_dict = scaler["X_full_std"]

# Numpy arrays aligned to feature order used in the notebook
X_mean = np.array([x_mean_dict[f] for f in X_SELECTED_FEATURES if f in x_mean_dict], dtype=float)
X_std = np.array([x_std_dict[f] for f in X_SELECTED_FEATURES if f in x_std_dict], dtype=float)
X_full_mean = np.array([x_full_mean_dict[f] for f in X_FULL_FEATURES if f in x_full_mean_dict], dtype=float)
X_full_std = np.array([x_full_std_dict[f] for f in X_FULL_FEATURES if f in x_full_std_dict], dtype=float)

# Well-formatted matrices
x_stats_matrix = pd.DataFrame(
    {
        "X_mean": pd.Series(x_mean_dict),
        "X_std": pd.Series(x_std_dict),
    }
)

x_full_stats_matrix = pd.DataFrame(
    {
        "X_full_mean": pd.Series(x_full_mean_dict),
        "X_full_std": pd.Series(x_full_std_dict),
    }
)

print(f"\nX stats matrix shape: {x_stats_matrix.shape}")
display(x_stats_matrix.round(6))

print(f"\nX full stats matrix shape: {x_full_stats_matrix.shape}")
display(x_full_stats_matrix.round(6))

In [15]:
# Cell 7: Build Ensemble Predictions
print("=" * 60)
print("BUILDING ENSEMBLE PREDICTIONS FROM TOP 3 MODELS")
print("=" * 60)

model_predictions = {}

for name, model_data in MODELS_1h.items():
    model = model_data["model"]
    feature_set = model_data["feature_set"]
    model_type = model_data["type"]

    # Select features + matching scaler stats
    if feature_set == "Full":
        features_to_use = X_FULL_FEATURES
        mean_dict = x_full_mean_dict
        std_dict = x_full_std_dict
    else:
        features_to_use = X_SELECTED_FEATURES
        mean_dict = x_mean_dict
        std_dict = x_std_dict

    # Check feature availability in data
    available_features = [f for f in features_to_use if f in DF_1H_MAX.columns]
    missing_features = [f for f in features_to_use if f not in DF_1H_MAX.columns]
    if missing_features:
        print(f"⚠️ {name}: Missing features in DF_1H_MAX -> {missing_features}")
        continue

    # Check feature availability in scaler stats
    missing_stats = [f for f in available_features if f not in mean_dict or f not in std_dict]
    if missing_stats:
        print(f"⚠️ {name}: Missing scaler stats -> {missing_stats}")
        continue

    X_model = DF_1H_MAX[available_features].copy()

    # Standardize with aligned mean/std by column name
    mean_s = pd.Series(mean_dict).reindex(available_features).astype(float)
    std_s = pd.Series(std_dict).reindex(available_features).astype(float).replace(0, np.nan)

    X_z = (X_model - mean_s) / std_s
    X_z = X_z.replace([np.inf, -np.inf], np.nan).dropna()

    if X_z.empty:
        print(f"⚠️ {name}: No rows left after standardization.")
        continue

    try:
        is_lstm = (
            hasattr(model, "input_shape")
            or str(type(model).__module__).startswith("tensorflow")
            or str(type(model).__module__).startswith("keras")
        )

        if is_lstm:
            try:
                X_z_reshaped = X_z.values.reshape((X_z.shape[0], 1, X_z.shape[1]))
                if hasattr(model, "build") and not model.built:
                    model.build(input_shape=(None, 1, X_z.shape[1]))
                proba = model.predict(X_z_reshaped, verbose=0, batch_size=128)
            except Exception as reshape_error:
                print(f"  ⚠️ {name}: 3D reshape failed, trying 2D: {reshape_error}")
                proba = model.predict(X_z.values, verbose=0, batch_size=128)

            if len(proba.shape) > 1 and proba.shape[1] > 1:
                prob_up_model = proba[:, 1]
            else:
                prob_up_model = proba.flatten()
        else:
            if hasattr(model, "predict_proba"):
                proba = model.predict_proba(X_z)
                prob_up_model = proba[:, 1]
            else:
                prob_up_model = model.predict(X_z)

        model_predictions[name] = pd.Series(prob_up_model, index=X_z.index)
        print(f"✓ {name}: Generated {len(prob_up_model)} predictions (mean: {np.mean(prob_up_model):.4f})")

    except Exception as e:
        print(f"✗ {name}: Error - {e}")

# Build ensemble by averaging predictions (using aligned indices)
if model_predictions:
    common_pred_index = model_predictions[list(model_predictions.keys())[0]].index
    for preds in model_predictions.values():
        common_pred_index = common_pred_index.intersection(preds.index)

    aligned_predictions = pd.DataFrame(
        {name: preds.loc[common_pred_index] for name, preds in model_predictions.items()}
    )

    prob_up_ensemble = aligned_predictions.mean(axis=1).values

    PREDICTIONS_1H_DF = DF_1H_MAX.loc[common_pred_index].copy()
    PREDICTIONS_1H_DF["prob_up"] = prob_up_ensemble
    PREDICTIONS_1H_DF["signal"] = (PREDICTIONS_1H_DF["prob_up"] > 0.5).astype(int)

    print("\n=== Ensemble Summary ===")
    print(f"Models used: {list(model_predictions.keys())}")
    print(f"Total predictions: {len(PREDICTIONS_1H_DF)}")
    print(f"Signal distribution: {PREDICTIONS_1H_DF['signal'].value_counts().to_dict()}")
else:
    print("ERROR: No models generated predictions!")

BUILDING ENSEMBLE PREDICTIONS FROM TOP 3 MODELS
  ⚠️ CNN_Full: 3D reshape failed, trying 2D: Exception encountered when calling Sequential.call().

Input 0 with name 'None' of layer 'conv1d_6' is incompatible with the layer: expected axis -1 of input shape to have value 1, but received input with shape (128, 1, 31)

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(128, 1, 31), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>
✓ CNN_Full: Generated 1310 predictions (mean: 0.5162)

=== Ensemble Summary ===
Models used: ['CNN_Full']
Total predictions: 1310
Signal distribution: {1: 962, 0: 348}


In [16]:
# Cell 8: Summary Statistics
print("=" * 60)
print("PREDICTION SUMMARY STATISTICS")
print("=" * 60)

print(f"\nPrediction Probability Distribution:")
print(PREDICTIONS_1H_DF['prob_up'].describe())

print(f"\nSignal Counts:")
print(f"  Buy signals (prob > 0.5):  {(PREDICTIONS_1H_DF['signal'] == 1).sum()}")
print(f"  Sell signals (prob <= 0.5): {(PREDICTIONS_1H_DF['signal'] == 0).sum()}")

print(f"\nRecent Predictions (last 10 bars):")
print(PREDICTIONS_1H_DF[['Close', 'prob_up', 'signal']].tail(10))

PREDICTION SUMMARY STATISTICS

Prediction Probability Distribution:
count    1310.000000
mean        0.516219
std         0.038867
min         0.106316
25%         0.498545
50%         0.516119
75%         0.533395
max         0.770345
Name: prob_up, dtype: float64

Signal Counts:
  Buy signals (prob > 0.5):  962
  Sell signals (prob <= 0.5): 348

Recent Predictions (last 10 bars):
Price                            Close   prob_up  signal
Datetime                                                
2026-01-27 13:00:00+00:00  2907.791260  0.505943       1
2026-01-27 14:00:00+00:00  2915.189209  0.495731       0
2026-01-27 15:00:00+00:00  2934.167969  0.510337       1
2026-01-27 16:00:00+00:00  2986.677002  0.518264       1
2026-01-27 17:00:00+00:00  2924.962158  0.562141       1
2026-01-27 18:00:00+00:00  2977.871094  0.518362       1
2026-01-27 19:00:00+00:00  2975.716309  0.508692       1
2026-01-27 20:00:00+00:00  3017.770508  0.464106       0
2026-01-27 21:00:00+00:00  3011.414062  0.385

In [19]:
def run_strategy_with_stop_loss(df, buy_threshold=0.6, sell_threshold=0.2, 
                                  stop_loss_pct=0.01, initial_balance=1000, 
                                  trading_cost=0.001):
    """
    Run strategy with stop loss taking priority over sell signals.
    Properly tracks units and balance.
    """
    balance = initial_balance
    units = 0  # Number of units owned
    position = 0  # 0 = no position, 1 = long
    entry_price = 0
    trades = []
    
    for i in range(len(df)):
        row = df.iloc[i]
        current_price = row['Close']
        prob = row['prob_up']
        low_price = row['Low']
        
        # If in position, check stop loss FIRST (priority)
        if position == 1:
            stop_price = entry_price * (1 - stop_loss_pct)
            
            # Stop loss triggered (using Low price for intrabar check)
            if low_price <= stop_price:
                exit_price = stop_price
                proceeds = units * exit_price * (1 - trading_cost)
                trades.append({
                    'type': 'STOP_LOSS',
                    'entry': entry_price,
                    'exit': exit_price,
                    'return': (exit_price / entry_price) - 1,
                    'balance': proceeds
                })
                balance = proceeds
                position = 0
                entry_price = 0
                units = 0
                continue  # Skip to next bar after stop loss
            
            # Sell signal (only if not stopped out)
            elif prob < sell_threshold:
                exit_price = current_price
                proceeds = units * exit_price * (1 - trading_cost)
                trades.append({
                    'type': 'SELL',
                    'entry': entry_price,
                    'exit': exit_price,
                    'return': (exit_price / entry_price) - 1,
                    'balance': proceeds
                })
                balance = proceeds
                position = 0
                entry_price = 0
                units = 0
        
        # Buy signal (only if not in position)
        elif position == 0 and prob > buy_threshold:
            entry_price = current_price * (1 + trading_cost)
            units = balance / entry_price  # Calculate units we can buy
            position = 1
            balance = 0  # All cash is now invested
    
    # Close any open position at end
    if position == 1:
        exit_price = df.iloc[-1]['Close']
        balance = units * exit_price * (1 - trading_cost)
    
    return balance, trades

# Test with default parameters
final_balance, trades = run_strategy_with_stop_loss(
    PREDICTIONS_1H_DF,
    buy_threshold=0.60,
    sell_threshold=0.20,
    stop_loss_pct=0.01,
    initial_balance=1000,
    trading_cost=0.001
)

print("=" * 60)
print("STRATEGY RESULTS (Default Parameters)")
print("=" * 60)
print(f"Initial Balance: $1,000.00")
print(f"Final Balance:   ${final_balance:.2f}")
print(f"Total Return:    {((final_balance / 1000) - 1) * 100:+.2f}%")
print(f"Total Trades:    {len(trades)}")

if trades:
    stop_losses = [t for t in trades if t['type'] == 'STOP_LOSS']
    sells = [t for t in trades if t['type'] == 'SELL']
    print(f"  - Stop Losses: {len(stop_losses)}")
    print(f"  - Regular Sells: {len(sells)}")

STRATEGY RESULTS (Default Parameters)
Initial Balance: $1,000.00
Final Balance:   $2843.90
Total Return:    +184.39%
Total Trades:    12
  - Stop Losses: 6
  - Regular Sells: 6


In [20]:
# Cell 10: Grid Search Optimization
print("=" * 60)
print("GRID SEARCH OPTIMIZATION")
print("=" * 60)

# Parameter grids
buy_thresholds = np.linspace(0.50, 0.80, 16)
sell_thresholds = np.linspace(0.20, 0.49, 15)
stop_losses = np.linspace(0.01, 0.05, 9)

print(f"Buy thresholds: {len(buy_thresholds)} values from {buy_thresholds[0]:.2f} to {buy_thresholds[-1]:.2f}")
print(f"Sell thresholds: {len(sell_thresholds)} values from {sell_thresholds[0]:.2f} to {sell_thresholds[-1]:.2f}")
print(f"Stop losses: {len(stop_losses)} values from {stop_losses[0]:.2%} to {stop_losses[-1]:.2%}")
print(f"Total combinations: {len(buy_thresholds) * len(sell_thresholds) * len(stop_losses)}")

results = []
total_combos = len(buy_thresholds) * len(sell_thresholds) * len(stop_losses)
combo_count = 0

for buy_t in buy_thresholds:
    for sell_t in sell_thresholds:
        for sl in stop_losses:
            combo_count += 1
            if combo_count % 500 == 0:
                print(f"Progress: {combo_count}/{total_combos} ({100*combo_count/total_combos:.1f}%)")
            
            final_bal, trades = run_strategy_with_stop_loss(
                PREDICTIONS_1H_DF,
                buy_threshold=buy_t,
                sell_threshold=sell_t,
                stop_loss_pct=sl,
                initial_balance=1000,
                trading_cost=0.001
            )
            
            results.append({
                'buy_threshold': buy_t,
                'sell_threshold': sell_t,
                'stop_loss': sl,
                'final_balance': final_bal,
                'return_pct': (final_bal / 1000 - 1) * 100,
                'num_trades': len(trades)
            })

# Find best result
results_df = pd.DataFrame(results)
best_idx = results_df['final_balance'].idxmax()
best = results_df.loc[best_idx]

print("\n" + "=" * 60)
print("OPTIMAL PARAMETERS FOUND")
print("=" * 60)
print(f"Buy Threshold:   {best['buy_threshold']:.4f}")
print(f"Sell Threshold:  {best['sell_threshold']:.4f}")
print(f"Stop Loss:       {best['stop_loss']:.4f} ({best['stop_loss']*100:.2f}%)")
print(f"Final Balance:   ${best['final_balance']:.2f}")
print(f"Total Return:    {best['return_pct']:+.2f}%")
print(f"Number of Trades: {best['num_trades']:.0f}")

GRID SEARCH OPTIMIZATION
Buy thresholds: 16 values from 0.50 to 0.80
Sell thresholds: 15 values from 0.20 to 0.49
Stop losses: 9 values from 1.00% to 5.00%
Total combinations: 2160
Progress: 500/2160 (23.1%)
Progress: 1000/2160 (46.3%)
Progress: 1500/2160 (69.4%)
Progress: 2000/2160 (92.6%)

OPTIMAL PARAMETERS FOUND
Buy Threshold:   0.7200
Sell Threshold:  0.4900
Stop Loss:       0.0100 (1.00%)
Final Balance:   $3116.85
Total Return:    +211.69%
Number of Trades: 1
